In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\HCAD All Address and Owner Roll test\2026HCADroll.csv",
    sep=",",
    dtype="string",
    encoding="utf-8-sig",
    low_memory=False
)

print(df.shape)
print(df.columns[:20])

(1048575, 71)
Index(['acct', 'yr', 'mailto', 'mail_addr_1', 'mail_addr_2', 'mail_city',
       'mail_state', 'mail_zip', 'mail_country', 'undeliverable', 'str_pfx',
       'str_num', 'str_num_sfx', 'str', 'str_sfx', 'str_sfx_dir', 'str_unit',
       'site_addr_1', 'site_addr_2', 'site_addr_3'],
      dtype='object')


In [2]:
n_rows = len(df)
n_acct = df["acct"].nunique(dropna=True)
print("rows:", n_rows, "unique acct:", n_acct, "dup rows:", n_rows - n_acct)

rows: 1048575 unique acct: 1048575 dup rows: 0


In [3]:
import re
from pathlib import Path

import pandas as pd
import duckdb

# -----------------------------
# 0) Project paths (ABSOLUTE)
# -----------------------------
PROJECT = Path(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate")
ROLL_CSV = PROJECT / r"HCAD All Address and Owner Roll test\2026HCADroll.csv"

GOLD_DIR = PROJECT / "gold"
WAREHOUSE_DIR = PROJECT / "warehouse"
DB_PATH = WAREHOUSE_DIR / "realestate.duckdb"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

ANCHOR_PARQUET = GOLD_DIR / "property_anchor.parquet"

print("PROJECT:", PROJECT)
print("ROLL_CSV:", ROLL_CSV)
print("ANCHOR_PARQUET:", ANCHOR_PARQUET)
print("DB_PATH:", DB_PATH)

# -----------------------------
# 1) Load appraisal roll
# -----------------------------
df = pd.read_csv(
    ROLL_CSV,
    sep=",",
    dtype="string",
    encoding="utf-8-sig",
    low_memory=False
)

print("Loaded roll:", df.shape)
print(df.columns[:20])

# -----------------------------
# 2) Build anchor (1NF)
# -----------------------------
SUFFIX_MAP = {
    "STREET":"ST", "ROAD":"RD", "AVENUE":"AVE", "BOULEVARD":"BLVD", "DRIVE":"DR", "LANE":"LN",
    "COURT":"CT", "CIRCLE":"CIR", "PLACE":"PL", "PARKWAY":"PKWY", "HIGHWAY":"HWY", "FREEWAY":"FWY"
}

def norm_addr(s: str) -> str:
    if s is None or pd.isna(s):
        return ""
    s = str(s).upper()
    s = re.sub(r"[^A-Z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = s.split()
    toks = [SUFFIX_MAP.get(t, t) for t in toks]
    return " ".join(toks)

def drop_unit(s: str) -> str:
    if not s:
        return s
    s = re.sub(r"\b(?:APT|UNIT|STE|SUITE|#)\s*\w+\b", "", s)  # non-capturing
    s = re.sub(r"\s+", " ", s).strip()
    return s

def join_parts(*parts):
    parts = [p for p in parts if p is not None and pd.notna(p) and str(p).strip() != ""]
    return " ".join(str(p).strip() for p in parts).strip()

df["acct"] = df["acct"].str.strip()
df["yr"] = pd.to_numeric(df["yr"], errors="coerce")

df["situs_full"] = df.apply(lambda r: join_parts(r.get("site_addr_1"), r.get("site_addr_2"), r.get("site_addr_3")), axis=1)
df["mail_full"]  = df.apply(lambda r: join_parts(r.get("mail_addr_1"), r.get("mail_addr_2"), r.get("mail_city"), r.get("mail_state"), r.get("mail_zip")), axis=1)

df["addr_key"] = df["site_addr_1"].map(norm_addr)
df["addr_key_no_unit"] = df["addr_key"].map(drop_unit)

owner = df["mailto"].fillna("").str.upper()
df["owner_entity_flag"] = owner.str.contains(
    r"\b(?:LLC|INC|CORP|CO|LTD|LP|LLP|TRUST|BANK|CITY|COUNTY)\b",
    regex=True
)

situs_city = df["site_addr_2"].fillna("").str.upper()
mail_city  = df["mail_city"].fillna("").str.upper()
df["absentee_owner_flag"] = (mail_city != "") & (situs_city != "") & (mail_city != situs_city)

df = df.sort_values(["acct", "yr"])
anchor = df.drop_duplicates(subset=["acct"], keep="last").copy()

anchor_cols = [
    "acct","yr",
    "site_addr_1","site_addr_2","site_addr_3",
    "str_pfx","str_num","str_num_sfx","str","str_sfx","str_sfx_dir","str_unit",
    "mailto","mail_addr_1","mail_addr_2","mail_city","mail_state","mail_zip","mail_country","undeliverable",
    "situs_full","mail_full",
    "addr_key","addr_key_no_unit",
    "owner_entity_flag","absentee_owner_flag",
]
anchor_cols = [c for c in anchor_cols if c in anchor.columns]
anchor = anchor[anchor_cols]

print("ANCHOR shape:", anchor.shape)
print("Unique acct:", anchor["acct"].nunique())

anchor.to_parquet(ANCHOR_PARQUET, index=False)
print("Saved anchor ->", ANCHOR_PARQUET)

# -----------------------------
# 3) Load anchor into DuckDB + build clean situs_zip + addr_zip_key
# -----------------------------
con = duckdb.connect(str(DB_PATH))

# Reload from parquet (authoritative)
con.execute("""
CREATE OR REPLACE TABLE property_anchor AS
SELECT * FROM read_parquet(?)
""", [str(ANCHOR_PARQUET)])

# Ensure columns exist
con.execute("ALTER TABLE property_anchor ADD COLUMN IF NOT EXISTS situs_zip VARCHAR")
con.execute("ALTER TABLE property_anchor ADD COLUMN IF NOT EXISTS addr_zip_key VARCHAR")

# ---- CLEAN REBUILD (prevents weird duplications) ----
con.execute("""
UPDATE property_anchor
SET situs_zip = REGEXP_EXTRACT(site_addr_3, '(\\d{5})', 1)
""")

con.execute("""
UPDATE property_anchor
SET addr_zip_key = addr_key || '|' || COALESCE(situs_zip,'')
""")

# quick check for bad keys (should be 0 rows)
bad = con.execute("""
SELECT acct, addr_key, site_addr_3, situs_zip, addr_zip_key
FROM property_anchor
WHERE addr_zip_key LIKE '%|%|%'
   OR addr_zip_key LIKE '%7700277002%'
LIMIT 20
""").fetchdf()

print("Bad addr_zip_key rows found:", len(bad))
if len(bad):
    print(bad)

# indexes
con.execute("CREATE INDEX IF NOT EXISTS idx_anchor_acct ON property_anchor(acct)")
con.execute("CREATE INDEX IF NOT EXISTS idx_anchor_addr_key ON property_anchor(addr_key)")
con.execute("CREATE INDEX IF NOT EXISTS idx_anchor_addr_zip_key ON property_anchor(addr_zip_key)")

print(con.execute("SELECT acct, addr_key, situs_zip, addr_zip_key FROM property_anchor LIMIT 5").fetchdf())
print(con.execute("SELECT COUNT(*) AS n, COUNT(DISTINCT acct) AS n_acct FROM property_anchor").fetchdf())

con.close()
print("Done.")

PROJECT: C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate
ROLL_CSV: C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\HCAD All Address and Owner Roll test\2026HCADroll.csv
ANCHOR_PARQUET: C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\property_anchor.parquet
DB_PATH: C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\warehouse\realestate.duckdb
Loaded roll: (1048575, 71)
Index(['acct', 'yr', 'mailto', 'mail_addr_1', 'mail_addr_2', 'mail_city',
       'mail_state', 'mail_zip', 'mail_country', 'undeliverable', 'str_pfx',
       'str_num', 'str_num_sfx', 'str', 'str_sfx', 'str_sfx_dir', 'str_unit',
       'site_addr_1', 'site_addr_2', 'site_addr_3'],
      dtype='object')
ANCHOR shape: (1048575, 26)
Unique acct: 1048575
Saved anchor -> C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\gold\property_anchor.parquet
Bad addr_zip_key rows found: 0
            acct         addr_key situs_zip           addr_zip_key
0  001001000